# 03 · IVP-Kiwi – Producto Final Unificado
### KiwiTrackingIA · Lote "El Abrojito" · 2019–2025

**Descripción:**  
Integración completa de todas las capas de análisis del proyecto:
- KVPI multianual (Sentinel-2)
- Topografía real (DEM Copernicus)
- GDD (Grados Día de Crecimiento, Tbase=10°C)
- Alertas climáticas (Isolation Forest – ML)
- Recomendaciones de manejo por planta (percentil p75 de estabilidad)
- Dashboard interactivo unificado (Folium)

**Inputs requeridos:**
- `datos/KVPI_serie_2019_2025.csv`
- `datos/muestreo_sistematico.csv`
- `datos/datos_climaticos_diarios_2019_2026.csv`
- `datos/Dataset_Kiwi_DL_Final_2019_2025.csv`
- `datos/DEM_Miramar_UTM.tif`
- `datos/Slope_Recalc_UTM.tif`

**Outputs generados:**
- `resultados/IVP_Kiwi_Producto_Final.html` — mapa interactivo unificado
- `resultados/datos_plantas_completos.csv`
- `resultados/gdd_por_campana.csv`
- `resultados/meses_anomalos.csv`
- `resultados/gdd_por_campana.png`
- `resultados/kvpi_vs_topografia.png`
- `resultados/anomalias_climaticas.png`
- `resultados/metadata_ejecucion.txt`


## 0 · Metadatos de ejecución

In [ ]:
# ============================================================
# KiwiTrackingIA – 03_IVP_Kiwi_Producto_Final.ipynb
# Autor: Darío Nicolás Sánchez Leguizamón
# Fecha: 2026-03-30
# Licencia: MIT
# Repositorio: https://github.com/sdarionicolas-boop/KiwiTrackingIA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import json
import os
from shapely.geometry import Point
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import folium
from folium import plugins
import branca.colormap as cm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

fecha_ejecucion = datetime.now()
version         = "2.0"
nombre_analisis = "IVP-KIWI - Producto Final Unificado"

print("=" * 80)
print(nombre_analisis)
print("=" * 80)
print(f"Versión : {version}")
print(f"Inicio  : {fecha_ejecucion.strftime('%Y-%m-%d %H:%M:%S')}")

## 1 · Configuración y carga de datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH  = '/content/drive/MyDrive/KiwiTrackingIA'
OUTPUT_PATH = f'{DRIVE_PATH}/resultados'
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("\n📊 Cargando datos...")

# 1.1 Serie KVPI (generada por notebook 01)
df_kvpi = pd.read_csv(f'{DRIVE_PATH}/KVPI_serie_2019_2025.csv')
print(f"✅ KVPI         : {len(df_kvpi)} registros")

# 1.2 Puntos productivos
df_pts = pd.read_csv(f'{DRIVE_PATH}/muestreo_sistematico.csv')
print(f"✅ Puntos        : {len(df_pts)} plantas")

# 1.3 Datos climáticos diarios (INTA o fuente local)
df_clima = pd.read_csv(f'{DRIVE_PATH}/datos_climaticos_diarios_2019_2026.csv')
df_clima['date'] = pd.to_datetime(df_clima['date'])
print(f"✅ Clima         : {len(df_clima)} registros diarios")

# 1.4 NDVI mensual (para Isolation Forest)
df_ndvi = pd.read_csv(f'{DRIVE_PATH}/Dataset_Kiwi_DL_Final_2019_2025.csv')
df_ndvi['date'] = pd.to_datetime(df_ndvi['date'])
print(f"✅ NDVI mensual  : {len(df_ndvi)} registros")

## 2 · Procesamiento KVPI

In [ ]:
def extraer_coordenadas(geo_str):
    """Extrae geometría Point desde columna .geo de GEE."""
    geo = json.loads(geo_str.replace('""', '"'))
    return Point(geo['coordinates'])

gdf_kvpi = gpd.GeoDataFrame(
    df_kvpi,
    geometry=df_kvpi[".geo"].apply(extraer_coordenadas),
    crs="EPSG:4326"
)

# KVPI medio, std y número de campañas por planta
kvpi_por_planta = (
    df_kvpi.groupby('id')['KVPI']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
kvpi_por_planta.columns = ['id', 'KVPI_mean', 'KVPI_std', 'n_campanas']

gdf_plantas = gpd.GeoDataFrame(
    kvpi_por_planta.merge(df_pts[['id', 'X', 'Y']], on='id', how='left'),
    geometry=gpd.points_from_xy(df_pts['X'], df_pts['Y']),
    crs="EPSG:4326"
)

print(f"✅ KVPI medio por planta: {len(gdf_plantas)} plantas")
print(f"   KVPI global: {gdf_plantas['KVPI_mean'].mean():.3f} ± {gdf_plantas['KVPI_mean'].std():.3f}")

## 3 · Topografía real (DEM Copernicus)

In [ ]:
print("⛰️  Cargando topografía real...")

try:
    dem_path   = f"{DRIVE_PATH}/DEM_Miramar_UTM.tif"
    slope_path = f"{DRIVE_PATH}/Slope_Recalc_UTM.tif"

    gdf_utm = gdf_plantas.to_crs("EPSG:32721")
    dem     = rasterio.open(dem_path)
    slope   = rasterio.open(slope_path)

    coords = [(g.x, g.y) for g in gdf_utm.geometry]
    gdf_utm['elev_m']    = [v[0] for v in dem.sample(coords)]
    gdf_utm['slope_pct'] = [v[0] for v in slope.sample(coords)]

    # Pasar de vuelta a WGS84
    gdf_plantas = gdf_utm.to_crs("EPSG:4326")
    gdf_plantas['elev_m']    = gdf_utm['elev_m']
    gdf_plantas['slope_pct'] = gdf_utm['slope_pct']

    print(f"✅ Elevación: {gdf_plantas['elev_m'].min():.1f} – {gdf_plantas['elev_m'].max():.1f} m")
    print(f"✅ Pendiente: {gdf_plantas['slope_pct'].min():.1f} – {gdf_plantas['slope_pct'].max():.1f} %")

except Exception as e:
    print(f"⚠️  Error cargando topografía: {e}")
    print("   Se usarán valores 0 para elev_m y slope_pct.")
    gdf_plantas['elev_m']    = 0
    gdf_plantas['slope_pct'] = 0

## 4 · Cálculo de GDD (Grados Día de Crecimiento, Tbase = 10°C)

In [ ]:
print("🌡️  Calculando GDD...")

df_clima['Tmedia']     = (df_clima['Tmax'] + df_clima['Tmin']) / 2
df_clima['gdd_diario'] = np.maximum(df_clima['Tmedia'] - 10, 0)

# Asignar temporada agrícola (inicia en julio)
df_clima['temporada'] = np.where(
    df_clima['date'].dt.month >= 7,
    df_clima['date'].dt.year,
    df_clima['date'].dt.year - 1
)

gdd_por_temporada = (
    df_clima.groupby('temporada')['gdd_diario']
    .sum().reset_index()
    .rename(columns={'gdd_diario': 'GDD_total'})
)

print(f"✅ GDD calculado")
print(f"   GDD total período      : {df_clima['gdd_diario'].sum():.0f} °C día")
print(f"   GDD promedio / campaña : {gdd_por_temporada['GDD_total'].mean():.0f} °C día")
display(gdd_por_temporada)

## 5 · Clasificación de ambientes productivos (terciles de KVPI)

In [ ]:
q33 = gdf_plantas['KVPI_mean'].quantile(0.33)
q66 = gdf_plantas['KVPI_mean'].quantile(0.66)

def clasificar_ambiente(kvpi):
    if kvpi >= q66:   return 'Alto Potencial'
    elif kvpi >= q33: return 'Transicional'
    else:             return 'Limitante'

gdf_plantas['ambiente'] = gdf_plantas['KVPI_mean'].apply(clasificar_ambiente)

print("📊 Distribución de ambientes:")
print(gdf_plantas['ambiente'].value_counts())
print(f"\n   Umbrales: Limitante ≤ {q33:.3f} | Transicional {q33:.3f}–{q66:.3f} | Alto ≥ {q66:.3f}")

## 6 · Alertas climáticas – Isolation Forest

In [ ]:
print("🤖 Detectando anomalías climáticas (Isolation Forest)...")

df_clima_mensual = (
    df_clima.groupby(df_clima['date'].dt.to_period('M'))
    .agg(Tmedia=('Tmedia','mean'), Precip=('Precip','sum'), GDD_mensual=('gdd_diario','sum'))
    .reset_index()
)
df_clima_mensual['date'] = df_clima_mensual['date'].dt.to_timestamp()

df_mensual = df_ndvi[['date','NDVI']].merge(df_clima_mensual, on='date', how='inner')
print(f"✅ Datos integrados: {len(df_mensual)} meses")

if len(df_mensual) > 0:
    features   = ['NDVI', 'Tmedia', 'Precip', 'GDD_mensual']
    df_features = df_mensual[features].dropna()

    scaler    = StandardScaler()
    X_scaled  = scaler.fit_transform(df_features)
    iso_forest = IsolationForest(contamination=0.1, random_state=42)
    df_mensual['anomalia']       = iso_forest.fit_predict(X_scaled)
    df_mensual['anomalia_score'] = iso_forest.score_samples(X_scaled)

    meses_anomalos = df_mensual[df_mensual['anomalia'] == -1].sort_values('anomalia_score')
    n_anomalos     = len(meses_anomalos)
    pct_anomalos   = n_anomalos / len(df_mensual) * 100

    print(f"\n✅ Meses anómalos: {n_anomalos} / {len(df_mensual)} ({pct_anomalos:.1f}%)")
    if n_anomalos > 0:
        print("\n🔍 Top 5 meses más anómalos:")
        for _, row in meses_anomalos.head(5).iterrows():
            print(f"   • {row['date'].strftime('%B %Y')}: NDVI={row['NDVI']:.3f}, "
                  f"GDD={row['GDD_mensual']:.0f}°C, Precip={row['Precip']:.0f}mm")
else:
    print("⚠️  Sin datos suficientes para Isolation Forest")
    meses_anomalos = pd.DataFrame()
    n_anomalos, pct_anomalos = 0, 0.0

## 7 · Recomendaciones estadísticas por planta

**Justificación metodológica del umbral p75 de estabilidad:**
- Enfoque conservador: sólo el 25% superior se considera "alta estabilidad"
- Minimiza falsos positivos en recomendaciones de manejo
- Permite focalizar recursos en plantas con mayor respuesta esperada
- Consistente con literatura de agricultura de precisión


In [ ]:
gdf_plantas['estabilidad'] = gdf_plantas['KVPI_mean'] / gdf_plantas['KVPI_std']
est_p75 = gdf_plantas['estabilidad'].quantile(0.75)

def recomendar_planta(row):
    kvpi  = row['KVPI_mean']
    estab = row['estabilidad']
    if   kvpi >= q66 and estab >= est_p75: return 'Mantener'
    elif kvpi <= q33 and estab >= est_p75: return 'Intervenir'
    else:                                  return 'Observar'

gdf_plantas['recomendacion'] = gdf_plantas.apply(recomendar_planta, axis=1)

print("📊 Distribución de recomendaciones:")
print(gdf_plantas['recomendacion'].value_counts())
print(f"\n   Umbral de estabilidad (p75): {est_p75:.1f}")

## 8 · Mapa interactivo unificado

In [ ]:
print("🗺️  Generando mapa interactivo unificado...")

colores_ambiente = {
    'Alto Potencial': '#2E7D32',
    'Transicional':   '#FFA000',
    'Limitante':      '#C62828'
}

m = folium.Map(
    location=[gdf_plantas.geometry.y.mean(), gdf_plantas.geometry.x.mean()],
    zoom_start=17,
    control_scale=True
)

# Heatmap de KVPI
heat_data = [[r.geometry.y, r.geometry.x, r['KVPI_mean']] for _, r in gdf_plantas.iterrows()]
plugins.HeatMap(heat_data, radius=12, blur=8, name='🔥 KVPI (Potencial)').add_to(m)

# Marcadores por planta
gdd_promedio = gdd_por_temporada['GDD_total'].mean()
for _, row in gdf_plantas.iterrows():
    color  = colores_ambiente.get(row['ambiente'], 'gray')
    popup  = f"""
    <div style="min-width:280px;font-family:monospace;">
    <b>🌱 Planta ID:</b> {int(row['id'])}<br>
    <b>🏞️ Ambiente:</b> <span style="color:{color};">{row['ambiente']}</span><br>
    <b>📊 KVPI medio:</b> {row['KVPI_mean']:.4f}<br>
    <b>📈 Estabilidad:</b> {row['estabilidad']:.1f}<br>
    <b>⛰️ Elevación:</b> {row['elev_m']:.1f} m<br>
    <b>📐 Pendiente:</b> {row['slope_pct']:.1f} %<br>
    <b>📅 Campañas:</b> {int(row['n_campanas'])}<br>
    <hr>
    <b>💡 RECOMENDACIÓN:</b> {row['recomendacion']}<br>
    <b>🌡️ GDD promedio campaña:</b> {gdd_promedio:.0f} °C día
    </div>
    """
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5, color=color, fill=True,
        fill_color=color, fill_opacity=0.85,
        tooltip=f"Planta {int(row['id'])} | {row['ambiente']} | {row['recomendacion']}",
        popup=folium.Popup(popup, max_width=380)
    ).add_to(m)

# Capas por ambiente
for amb, name in [('Alto Potencial','🟢 Alto Potencial'),
                  ('Transicional','🟠 Transicional'),
                  ('Limitante','🔴 Limitante')]:
    fg = folium.FeatureGroup(name=name)
    c  = colores_ambiente[amb]
    for _, r in gdf_plantas[gdf_plantas['ambiente']==amb].iterrows():
        folium.CircleMarker([r.geometry.y, r.geometry.x], radius=4,
                            color=c, fill=True, fill_color=c, fill_opacity=0.6).add_to(fg)
    fg.add_to(m)

# Satélite + minimap
folium.TileLayer('https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
                 attr='Google', name='🛰️ Satélite').add_to(m)
plugins.MiniMap(toggle_display=True, position='bottomright', width=150, height=150).add_to(m)

# Leyenda dinámica
mes_max = meses_anomalos.iloc[0]['date'].strftime('%B %Y') if n_anomalos > 0 else 'N/A'
legend_html = f"""
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;
            padding:12px 14px;border:2px solid #444;border-radius:8px;
            font-size:12px;max-width:320px;font-family:monospace;
            box-shadow:2px 2px 6px rgba(0,0,0,0.3);">
<b>🏞️ IVP-KIWI · Producto Final Unificado</b><br>
<span style="font-size:10px;">Lote "El Abrojito" · 2019–2025</span>
<hr>
<b>Ambientes (KVPI):</b><br>
<span style="color:#2E7D32;">● Alto Potencial</span> (KVPI ≥ {q66:.2f})<br>
<span style="color:#FFA000;">● Transicional</span> ({q33:.2f}–{q66:.2f})<br>
<span style="color:#C62828;">● Limitante</span> (≤ {q33:.2f})<br>
<hr>
<b>GDD (Tbase=10°C):</b><br>
• GDD promedio campaña: {gdd_promedio:.0f} °C día<br>
• Total período: {df_clima['gdd_diario'].sum():.0f} °C día<br>
<hr>
<b>Alertas climáticas (Isolation Forest):</b><br>
• {n_anomalos} meses anómalos ({pct_anomalos:.1f}%) · Mes pico: {mes_max}<br>
<hr>
<b>Recomendaciones (umbral estab. p75={est_p75:.1f}):</b><br>
• Mantener: {len(gdf_plantas[gdf_plantas['recomendacion']=='Mantener'])} &nbsp;
  Observar: {len(gdf_plantas[gdf_plantas['recomendacion']=='Observar'])} &nbsp;
  Intervenir: {len(gdf_plantas[gdf_plantas['recomendacion']=='Intervenir'])}<br>
<hr>
<span style="font-size:10px;">📊 KVPI global: {gdf_plantas['KVPI_mean'].mean():.3f} ± {gdf_plantas['KVPI_mean'].std():.3f}<br>
🖱️ Hover → info rápida | Click → datos completos</span>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(position='topright', collapsed=False).add_to(m)

out_map = f"{OUTPUT_PATH}/IVP_Kiwi_Producto_Final.html"
m.save(out_map)
print(f"\n✅ Mapa guardado: {out_map}")
m

## 9 · Gráficos de apoyo

In [ ]:
# --- Gráfico 1: GDD por campaña ---
fig, ax = plt.subplots(figsize=(12, 6))
campanas   = gdd_por_temporada['temporada'].astype(str) + '/' + (gdd_por_temporada['temporada']+1).astype(str)
gdd_values = gdd_por_temporada['GDD_total']
bars = ax.bar(campanas, gdd_values, color='steelblue', alpha=0.75, edgecolor='black')
for bar, val in zip(bars, gdd_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xlabel('Campaña', fontsize=12)
ax.set_ylabel('GDD Total (°C día)', fontsize=12)
ax.set_title('Grados Día de Crecimiento (GDD) por Campaña\nTbase=10°C · Lote "El Abrojito"', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/gdd_por_campana.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ gdd_por_campana.png")

In [ ]:
# --- Gráfico 2: KVPI vs Topografía ---
colors_plot = [colores_ambiente[a] for a in gdf_plantas['ambiente']]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(gdf_plantas['elev_m'],    gdf_plantas['KVPI_mean'], c=colors_plot, alpha=0.7, s=50, edgecolors='black')
axes[0].set_xlabel('Elevación (m)');       axes[0].set_ylabel('KVPI medio')
axes[0].set_title('KVPI vs Elevación');    axes[0].grid(True, alpha=0.3)

axes[1].scatter(gdf_plantas['slope_pct'], gdf_plantas['KVPI_mean'], c=colors_plot, alpha=0.7, s=50, edgecolors='black')
axes[1].set_xlabel('Pendiente (%)');       axes[1].set_ylabel('KVPI medio')
axes[1].set_title('KVPI vs Pendiente');    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_PATH}/kvpi_vs_topografia.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ kvpi_vs_topografia.png")

In [ ]:
# --- Gráfico 3: Anomalías climáticas ---
if n_anomalos > 0:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df_mensual['date'], df_mensual['anomalia_score'], 'o-', color='gray', alpha=0.5, label='Todos los meses')
    ax.scatter(meses_anomalos['date'], meses_anomalos['anomalia_score'],
               color='red', s=80, zorder=5, label='Meses anómalos')
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('Fecha', fontsize=11)
    ax.set_ylabel('Puntaje de anomalía (Isolation Forest)', fontsize=11)
    ax.set_title('Detección de Anomalías Climáticas – Isolation Forest\n(Puntaje más negativo = mayor atipicidad)', fontsize=12)
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_PATH}/anomalias_climaticas.png", dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ anomalias_climaticas.png")
else:
    print("ℹ️  Sin meses anómalos: no se genera gráfico de anomalías.")

## 10 · Exportación de datos

In [ ]:
df_export = gdf_plantas[['id','KVPI_mean','KVPI_std','estabilidad',
                         'ambiente','recomendacion','elev_m','slope_pct','X','Y']].copy()
df_export.to_csv(f"{OUTPUT_PATH}/datos_plantas_completos.csv", index=False)
print("✅ datos_plantas_completos.csv")

gdd_por_temporada.to_csv(f"{OUTPUT_PATH}/gdd_por_campana.csv", index=False)
print("✅ gdd_por_campana.csv")

if n_anomalos > 0:
    meses_anomalos.to_csv(f"{OUTPUT_PATH}/meses_anomalos.csv", index=False)
    print("✅ meses_anomalos.csv")

## 11 · Metadatos de ejecución

In [ ]:
mes_max_str = meses_anomalos.iloc[0]['date'].strftime('%B %Y') if n_anomalos > 0 else 'N/A'

with open(f"{OUTPUT_PATH}/metadata_ejecucion.txt", "w", encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write(f"{nombre_analisis}\n")
    f.write("="*60 + "\n\n")
    f.write(f"Fecha     : {fecha_ejecucion.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Versión   : {version}\n\n")
    f.write("DATOS DE ENTRADA\n" + "-"*40 + "\n")
    f.write(f"KVPI registros    : {len(df_kvpi)}\n")
    f.write(f"Plantas           : {len(gdf_plantas)}\n")
    f.write(f"Clima (días)      : {len(df_clima)}\n")
    f.write(f"NDVI mensual      : {len(df_ndvi)}\n\n")
    f.write("RESULTADOS\n" + "-"*40 + "\n")
    f.write(f"KVPI medio global : {gdf_plantas['KVPI_mean'].mean():.3f} ± {gdf_plantas['KVPI_mean'].std():.3f}\n")
    f.write(f"GDD promedio      : {gdd_por_temporada['GDD_total'].mean():.0f} °C día\n")
    f.write(f"Meses anómalos    : {n_anomalos} / {len(df_mensual)} ({pct_anomalos:.1f}%)\n")
    f.write(f"Mes más anómalo   : {mes_max_str}\n\n")
    f.write("JUSTIFICACIÓN METODOLÓGICA\n" + "-"*40 + "\n")
    f.write(f"Umbral estabilidad (p75): {est_p75:.1f}\n")
    f.write("Clasificación de ambientes: terciles de KVPI (p33/p66)\n\n")
    f.write("="*60 + "\nFIN METADATOS\n" + "="*60 + "\n")

print(f"✅ metadata_ejecucion.txt")

## 12 · Resumen final

In [ ]:
print("\n" + "="*80)
print("RESUMEN FINAL – IVP-KIWI PRODUCTO UNIFICADO")
print("="*80)
print(f"""
📊 ESTADÍSTICAS DEL LOTE
  Plantas analizadas : {len(gdf_plantas)}
  KVPI medio global  : {gdf_plantas['KVPI_mean'].mean():.3f} ± {gdf_plantas['KVPI_mean'].std():.3f}

🏞️ AMBIENTES PRODUCTIVOS
  Alto Potencial (≥ {q66:.2f}) : {len(gdf_plantas[gdf_plantas['ambiente']=='Alto Potencial'])} plantas
  Transicional  ({q33:.2f}–{q66:.2f}) : {len(gdf_plantas[gdf_plantas['ambiente']=='Transicional'])} plantas
  Limitante     (≤ {q33:.2f}) : {len(gdf_plantas[gdf_plantas['ambiente']=='Limitante'])} plantas

🌡️ GDD (Tbase=10°C)
  GDD total período       : {df_clima['gdd_diario'].sum():.0f} °C día
  GDD promedio / campaña  : {gdd_por_temporada['GDD_total'].mean():.0f} °C día

🤖 ALERTAS CLIMÁTICAS (Isolation Forest)
  Meses anómalos : {n_anomalos} / {len(df_mensual)} ({pct_anomalos:.1f}%)

💡 RECOMENDACIONES (umbral p75 estabilidad = {est_p75:.1f})
  Mantener   : {len(gdf_plantas[gdf_plantas['recomendacion']=='Mantener'])} plantas
  Observar   : {len(gdf_plantas[gdf_plantas['recomendacion']=='Observar'])} plantas
  Intervenir : {len(gdf_plantas[gdf_plantas['recomendacion']=='Intervenir'])} plantas
""")
print("="*80)
print("✅ IVP-KIWI – PRODUCTO FINAL COMPLETADO")
print("="*80)